In [1]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Điền thông tin Kaggle API của bạn vào đây
# (Vào Kaggle -> Settings -> Creatxe New Token để lấy username và key)
os.environ['KAGGLE_USERNAME'] = "huyphngquc"
os.environ['KAGGLE_KEY'] = "d2051dddb134d7809d29527c977cab58"

# Tạo thư mục trên Drive để lưu dataset cố định (nếu chưa có)
DRIVE_DATASET_DIR = '/content/drive/MyDrive/Thesis_ViVQA_Data'
os.makedirs(DRIVE_DATASET_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
import os
import time

LOCAL_DATA_DIR = '/content/vivqa_dataset'
ZIP_PATH_ON_DRIVE = f'{DRIVE_DATASET_DIR}/vivqa-1.zip' # Đảm bảo biến này đúng
ZIP_PATH_LOCAL = '/content/vivqa-1.zip'

start_time = time.time()

# 1. Xóa rác cũ nếu có (Quan trọng để tránh giải nén đè lên file lỗi)
!rm -rf {LOCAL_DATA_DIR}
!rm -f {ZIP_PATH_LOCAL}

# 2. Copy bằng lệnh Linux (Trâu hơn shutil)
print("Đang copy từ Drive sang Local...")
!cp "{ZIP_PATH_ON_DRIVE}" "{ZIP_PATH_LOCAL}"

# 3. Giải nén và TỰ ĐỘNG SỬA LỖI (Dùng option -FF nếu cần)
print("Đang giải nén...")
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Thử giải nén bình thường trước
result = !unzip -q -n {ZIP_PATH_LOCAL} -d {LOCAL_DATA_DIR}

# Kiểm tra nếu vẫn còn báo lỗi "bad zipfile offset"
if any("bad zipfile offset" in line for line in result):
    print("⚠️ File zip có vẻ bị lỗi offset, đang thử fix...")
    # Lệnh fix file zip của linux
    !zip -F {ZIP_PATH_LOCAL} --out {ZIP_PATH_LOCAL}_fixed.zip
    !unzip -q -n {ZIP_PATH_LOCAL}_fixed.zip -d {LOCAL_DATA_DIR}

# 4. Dọn dẹp
if os.path.exists(ZIP_PATH_LOCAL): os.remove(ZIP_PATH_LOCAL)

print(f"✅ Xong! Kiểm tra thử: {len(os.listdir(LOCAL_DATA_DIR))} folders/files.")

✅ Xong! Kiểm tra thử: 13 folders/files.


In [3]:
# 4. Kiểm tra và in ra cấu trúc thực tế
print("\n--- KIỂM TRA CẤU TRÚC DATA ---")
if os.path.exists(LOCAL_DATA_DIR):
    # Liệt kê tất cả file và folder (kể cả trong folder con)
    all_files = []
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for name in files:
            all_files.append(os.path.join(root, name))
    
    print(f"📍 Path cơ sở: {LOCAL_DATA_DIR}")
    print(f"📦 Tổng số lượng file tìm thấy: {len(all_files)}")
    
    if len(all_files) > 0:
        print("📄 5 file đầu tiên để check path:")
        for f in all_files[:5]:
            print(f"   -> {f}")
    else:
        print("❌ CẢNH BÁO: Thư mục tồn tại nhưng không có file nào bên trong (do giải nén lỗi).")
else:
    print("❌ LỖI: Thư mục LOCAL_DATA_DIR không tồn tại.")


--- KIỂM TRA CẤU TRÚC DATA ---
📍 Path cơ sở: /content/vivqa_dataset
📦 Tổng số lượng file tìm thấy: 11720
📄 5 file đầu tiên để check path:
   -> /content/vivqa_dataset/merged_train.csv
   -> /content/vivqa_dataset/yolo_final_results.csv
   -> /content/vivqa_dataset/answer_space.txt
   -> /content/vivqa_dataset/Owl_train.csv
   -> /content/vivqa_dataset/answer_type_mapping.csv


In [4]:
!pip install -q transformers timm accelerate bitsandbytes scikit-learn underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 118.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.0 MB/s eta 0:00:00


In [5]:
import warnings, logging
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("torch.utils.data.dataloader").setLevel(logging.ERROR)

In [6]:
import os
import ast
import math
import torch

import numpy as np
import pandas as pd
import torch.nn as nn
from PIL import Image

from tqdm.auto import tqdm
import torch.nn.functional as F
from underthesea import word_tokenize
from sklearn.preprocessing import LabelEncoder

from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import Blip2Processor, Blip2Model, AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

In [7]:
# ==========================================
# 1. CẤU HÌNH (CONFIG)
# ==========================================
CONFIG = {
    "train_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/train.csv",
    "train_img_dir": "/content/vivqa_dataset/train",
    "train_box_csv": "/content/vivqa_dataset/merged_train.csv",

    "test_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/test.csv",
    "test_img_dir": "/content/vivqa_dataset/test",
    "test_box_csv": "/content/vivqa_dataset/vivqa_ot.csv", ## cố tính để path box sai để infer đừng bị leakage 

    "save_path": "/content/vivqa_dataset/vivqa_v100.pth",

    "blip_model": "Salesforce/blip2-opt-2.7b",
    "text_model": "vinai/phobert-base",

    "batch_size": 32,
    "epochs": 25,
    "lr": 5e-5,
    "lr_qformer": 1e-5,
    "weight_decay": 0.05,

    # CẬP NHẬT: Giảm trọng số SmoothL1, tăng mạnh trọng số GIoU để ép model học overlap thay vì tọa độ thô
    "lambda_bbox": 2.0,
    "lambda_qtype": 1.0,
    "lambda_giou": 10.0,
    
    "type_embed_dim": 64,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "max_length": 50,
    "patience": 5,
    "delta": 0.001,

    "box_required_types": [2],
    "qformer_unfreeze_layers": 2,
    "num_workers": 2
}


In [8]:
# ==========================================
# 2. HÀM BỔ TRỢ
# ==========================================
def preprocess_vietnamese(text):
    text = str(text).lower().strip()
    return word_tokenize(text, format="text")

def find_image_path(folder, img_id):
    valid_exts = ['.jpg', '.png', '.jpeg']
    img_id_str = str(img_id)
    if img_id_str.lower().endswith(tuple(valid_exts)):
        path = os.path.join(folder, img_id_str)
        if os.path.exists(path): return path
    for ext in valid_exts:
        path = os.path.join(folder, f"{img_id_str}{ext}")
        if os.path.exists(path): return path
    return None

def calculate_iou(pred_boxes, target_boxes):
    # Sửa lỗi diện tích âm khi model đoán ngược tọa độ
    p_left = torch.min(pred_boxes[:, 0], pred_boxes[:, 2])
    p_top = torch.min(pred_boxes[:, 1], pred_boxes[:, 3])
    p_right = torch.max(pred_boxes[:, 0], pred_boxes[:, 2])
    p_bottom = torch.max(pred_boxes[:, 1], pred_boxes[:, 3])

    inter_xmin = torch.max(p_left, target_boxes[:, 0])
    inter_ymin = torch.max(p_top, target_boxes[:, 1])
    inter_xmax = torch.min(p_right, target_boxes[:, 2])
    inter_ymax = torch.min(p_bottom, target_boxes[:, 3])

    inter_w = torch.clamp(inter_xmax - inter_xmin, min=0)
    inter_h = torch.clamp(inter_ymax - inter_ymin, min=0)
    inter_area = inter_w * inter_h

    pred_area = (p_right - p_left) * (p_bottom - p_top)
    target_area = (target_boxes[:, 2] - target_boxes[:, 0]) * (target_boxes[:, 3] - target_boxes[:, 1])
    union_area = pred_area + target_area - inter_area + 1e-6

    iou = inter_area / union_area
    return torch.clamp(iou, min=0.0, max=1.0)

def giou_loss(pred_boxes, target_boxes):
    # Lấy tọa độ an toàn cho pred_boxes (tránh lỗi xmin > xmax sinh ra diện tích âm)
    px1, py1, px2, py2 = pred_boxes[:, 0], pred_boxes[:, 1], pred_boxes[:, 2], pred_boxes[:, 3]
    tx1, ty1, tx2, ty2 = target_boxes[:, 0], target_boxes[:, 1], target_boxes[:, 2], target_boxes[:, 3]

    p_left = torch.min(px1, px2)
    p_right = torch.max(px1, px2)
    p_top = torch.min(py1, py2)
    p_bottom = torch.max(py1, py2)

    pw = (p_right - p_left) + 1e-6
    ph = (p_bottom - p_top) + 1e-6
    pred_area = pw * ph

    target_area = (tx2 - tx1) * (ty2 - ty1)

    # Tính vùng giao nhau (Intersection)
    inter_xmin = torch.max(p_left, tx1)
    inter_ymin = torch.max(p_top, ty1)
    inter_xmax = torch.min(p_right, tx2)
    inter_ymax = torch.min(p_bottom, ty2)

    inter_w = torch.clamp(inter_xmax - inter_xmin, min=0)
    inter_h = torch.clamp(inter_ymax - inter_ymin, min=0)
    inter_area = inter_w * inter_h

    union_area = pred_area + target_area - inter_area + 1e-6
    iou = inter_area / union_area

    # Tính hộp bao quanh (Enclosing box)
    enc_xmin = torch.min(p_left, tx1)
    enc_ymin = torch.min(p_top, ty1)
    enc_xmax = torch.max(p_right, tx2)
    enc_ymax = torch.max(p_bottom, ty2)

    enc_w = torch.clamp(enc_xmax - enc_xmin, min=0)
    enc_h = torch.clamp(enc_ymax - enc_ymin, min=0)
    enc_area = enc_w * enc_h + 1e-6

    giou = iou - (enc_area - union_area) / enc_area
    return (1 - giou).mean()


In [9]:
# ==========================================
# 3. DATASET
# ==========================================
class ViVQATripleTaskDataset(Dataset):
    def __init__(self, dataframe, image_dir, blip_processor, tokenizer, label_encoder, unk_token_id):
        self.data = dataframe
        self.image_dir = image_dir
        self.blip_processor = blip_processor
        self.tokenizer = tokenizer
        self.label_encoder = label_encoder
        self.unk_token_id = unk_token_id
        self.known_classes = set(label_encoder.classes_)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_id = row.get('img_id', row.get('image'))
        img_path = find_image_path(self.image_dir, img_id)
        try:
            image = Image.open(img_path).convert("RGB") if img_path else Image.new('RGB', (224, 224))
        except:
            image = Image.new('RGB', (224, 224))
        orig_w, orig_h = image.size
        pixel_values = self.blip_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)

        target_box = torch.zeros(4, dtype=torch.float32)
        has_box = torch.tensor(0.0, dtype=torch.float32)
        
        # CẬP NHẬT: Ưu tiên lấy merged_box, nếu không có mới lấy owl_box
        box_str = row.get('merged_box', row.get('owl_box', None))
        
        if pd.notna(box_str):
            try:
                box = ast.literal_eval(str(box_str))
                xmin, ymin, xmax, ymax = box
                target_box[0], target_box[1] = xmin / orig_w, ymin / orig_h
                target_box[2], target_box[3] = xmax / orig_w, ymax / orig_h
                target_box = torch.clamp(target_box, 0.0, 1.0)
                has_box = torch.tensor(1.0, dtype=torch.float32)
            except:
                pass

        question = preprocess_vietnamese(row['question'])
        text_encoding = self.tokenizer(
            question, return_tensors="pt", padding="max_length",
            truncation=True, max_length=CONFIG['max_length'], add_special_tokens=True
        )
        answer = str(row.get('answer', '')).lower().strip()
        label = self.label_encoder.transform([answer])[0] if answer in self.known_classes else self.unk_token_id
        q_type = int(row.get('type', 0))

        return {
            "pixel_values": pixel_values,
            "input_ids": text_encoding['input_ids'].squeeze(0),
            "attention_mask": text_encoding['attention_mask'].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
            "q_type": torch.tensor(q_type, dtype=torch.long),
            "target_box": target_box,
            "has_box": has_box
        }


In [10]:
# ==========================================
# 4. MODEL
# ==========================================
class CrossAttentionFusion(nn.Module):
    def __init__(self, visual_dim, text_dim, embed_dim, num_heads=8):
        super().__init__()
        self.vis_project = nn.Linear(visual_dim, embed_dim)
        self.txt_project = nn.Linear(text_dim, embed_dim)
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, visual_features, text_features):
        v_proj = self.vis_project(visual_features)
        t_proj = self.txt_project(text_features)
        attn_output, _ = self.multihead_attn(query=t_proj, key=v_proj, value=v_proj)
        combined = self.norm(attn_output + t_proj)
        return combined.mean(dim=1) + combined.max(dim=1)[0]

class TextGuidedBBoxHead(nn.Module):
    """
    BBox head cải tiến:
    - Deeper network với 3 layers
    - Tách nhánh: một nhánh tính từ attended visual, một nhánh từ text
    - Final prediction từ residual của hai nhánh
    """
    def __init__(self, visual_dim=768, text_dim=768, type_dim=64):
        super().__init__()
        self.attn_proj = nn.Linear(text_dim + type_dim, visual_dim)

        # Nhánh visual
        self.vis_branch = nn.Sequential(
            nn.Linear(visual_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
        )
        # Nhánh text+type
        self.txt_branch = nn.Sequential(
            nn.Linear(text_dim + type_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
        )
        # Regression head cuối
        self.regressor = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 4),
            nn.Sigmoid()
        )

    def forward(self, vis_seq, text_feat, type_feat):
        context = torch.cat([text_feat, type_feat], dim=-1)         # (B, 832)
        attn_query = self.attn_proj(context).unsqueeze(-1)           # (B, 768, 1)
        attn_weights = torch.bmm(vis_seq, attn_query).squeeze(-1)    # (B, 32)
        attn_weights = torch.softmax(attn_weights, dim=-1).unsqueeze(-1)  # (B, 32, 1)
        attended_vis = (vis_seq * attn_weights).sum(dim=1)            # (B, 768)

        vis_feat = self.vis_branch(attended_vis)   # (B, 256)
        txt_feat = self.txt_branch(context)        # (B, 256)
        combined = torch.cat([vis_feat, txt_feat], dim=-1)  # (B, 512)
        return self.regressor(combined)


In [11]:
class ViVQATripleTaskModel(nn.Module):
    def __init__(self, num_classes, num_q_types):
        super().__init__()
        print("⏳ Đang load BLIP-2 Vision & Q-Former...")
        tmp_blip = Blip2Model.from_pretrained(CONFIG['blip_model'], torch_dtype=torch.float16)
        self.vision_model = tmp_blip.vision_model
        self.qformer = tmp_blip.qformer
        self.query_tokens = tmp_blip.query_tokens
        del tmp_blip

        # Freeze toàn bộ vision_model
        for param in self.vision_model.parameters():
            param.requires_grad = False

        # Freeze toàn bộ Q-Former trước, rồi unfreeze N layers cuối
        # -> Cho bbox_head có gradient thực từ Q-Former
        for param in self.qformer.parameters():
            param.requires_grad = False
        self.query_tokens.requires_grad = False

        self._unfreeze_qformer_last_layers(CONFIG['qformer_unfreeze_layers'])

        print("⏳ Đang load PhoBERT...")
        self.phobert = AutoModel.from_pretrained(CONFIG['text_model'])

        embed_dim = 768
        self.fusion = CrossAttentionFusion(visual_dim=768, text_dim=768, embed_dim=embed_dim)
        self.q_type_head = nn.Linear(768, num_q_types)
        self.type_emb = nn.Embedding(num_q_types, CONFIG['type_embed_dim'])

        in_features = embed_dim + CONFIG['type_embed_dim']
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512), nn.LayerNorm(512), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(512, num_classes)
        )
        self.bbox_head = TextGuidedBBoxHead(visual_dim=768, text_dim=768, type_dim=CONFIG['type_embed_dim'])

    def _unfreeze_qformer_last_layers(self, n_layers):
        """
        Unfreeze n layers cuối của Q-Former encoder.
        Q-Former dùng BertEncoder với .encoder.layer list.
        Chỉ unfreeze layer norm và cross-attention để tiết kiệm memory.
        """
        try:
            layers = self.qformer.encoder.layer
            total = len(layers)
            unfreeze_from = max(0, total - n_layers)
            for i, layer in enumerate(layers):
                if i >= unfreeze_from:
                    for param in layer.parameters():
                        param.requires_grad = True
            # Unfreeze query_tokens để Q-Former có thể attend tốt hơn
            self.query_tokens.requires_grad = True
            n_unfrozen = sum(p.numel() for p in self.qformer.parameters() if p.requires_grad)
            print(f"✅ Đã unfreeze {n_layers} layers cuối Q-Former ({n_unfrozen/1e6:.1f}M params)")
        except AttributeError:
            print("⚠️  Không tìm thấy qformer.encoder.layer, bỏ qua unfreeze")

    def forward(self, pixel_values, input_ids, attention_mask):
        # Vision model vẫn no_grad để tiết kiệm VRAM
        with torch.no_grad():
            vision_outputs = self.vision_model(pixel_values=pixel_values)
            image_embeds = vision_outputs.last_hidden_state
            image_attention_mask = torch.ones(image_embeds.size()[:-1], dtype=torch.long, device=image_embeds.device)

        # Q-Former: các layer được unfreeze sẽ có gradient
        query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
        # Detach image_embeds để không lan truyền ngược về vision_model
        image_embeds_d = image_embeds.detach()
        query_outputs = self.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds_d,
            encoder_attention_mask=image_attention_mask,
            output_attentions=False
        )
        visual_seq = query_outputs.last_hidden_state  # (B, 32, 768) - có gradient

        text_outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        text_feats = text_outputs.last_hidden_state
        visual_seq = visual_seq.to(text_feats.dtype)

        cls_token_feats = text_feats[:, 0, :]
        q_type_logits = self.q_type_head(cls_token_feats)
        soft_q_type = torch.softmax(q_type_logits, dim=-1)
        type_vec = torch.matmul(soft_q_type, self.type_emb.weight)

        fused_vec = self.fusion(visual_seq, text_feats)
        final_features = torch.cat([fused_vec, type_vec], dim=-1)
        ans_logits = self.classifier(final_features)

        pred_boxes = self.bbox_head(visual_seq, cls_token_feats, type_vec)

        return ans_logits, pred_boxes, q_type_logits

class CosineDecay:
    def __init__(self, start_value, end_value, total_steps):
        self.start_value = start_value
        self.end_value = end_value
        self.total_steps = total_steps

    def get_value(self, current_step):
        step = min(current_step, self.total_steps)
        cosine_val = 1 + math.cos(math.pi * step / self.total_steps)
        return self.end_value + 0.5 * (self.start_value - self.end_value) * cosine_val

In [12]:
# ==========================================
# 5. TRAIN LOOP
# ==========================================
class EarlyStopping:
    def __init__(self, patience=5, delta=0, path='checkpoint.pth'):
        self.patience, self.delta, self.path = patience, delta, path
        self.counter, self.best_score, self.early_stop = 0, None, False
        self.best_epoch = 0 # Lưu lại epoch tốt nhất

    def __call__(self, current_score, model, epoch):
        if self.best_score is None:
            self.best_score = current_score
            self.save_checkpoint(model)
            self.best_epoch = epoch
        elif current_score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_score = current_score
            self.save_checkpoint(model)
            self.best_epoch = epoch
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)

def prepare_data():
    blip_processor = Blip2Processor.from_pretrained(CONFIG['blip_model'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model'], use_fast=False)

    df_raw = pd.read_csv(CONFIG['train_csv'])
    if 'Unnamed: 0' in df_raw.columns:
        df_raw.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)

    df_box = pd.read_csv(CONFIG['train_box_csv'])
    if 'Unnamed: 0' in df_box.columns:
        df_box.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)

    # Đổi thành merged_box
    df_merged = pd.merge(df_raw, df_box[['question_id', 'merged_box']], on='question_id', how='left')

    all_answers = df_merged['answer'].apply(lambda x: str(x).lower().strip()).unique().tolist()
    if 'unknown' not in all_answers: all_answers.append('unknown')
    label_encoder = LabelEncoder().fit(all_answers)
    unk_token_id = label_encoder.transform(['unknown'])[0]
    num_q_types = int(df_merged['type'].max() + 1)

    train_df, val_df = train_test_split(df_merged, test_size=0.2, random_state=42, stratify=df_merged['type'])
    print(f"Kích thước tập Train: {len(train_df)} | Tập Val (20%): {len(val_df)}")

    train_loader = DataLoader(
        ViVQATripleTaskDataset(train_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id),
        batch_size=CONFIG['batch_size'], shuffle=True,
        num_workers=CONFIG['num_workers'], persistent_workers=True, pin_memory=True
    )
    val_loader = DataLoader(
        ViVQATripleTaskDataset(val_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id),
        batch_size=CONFIG['batch_size'], shuffle=False,
        num_workers=CONFIG['num_workers'], persistent_workers=True, pin_memory=True
    )

    return train_loader, val_loader, label_encoder, num_q_types, blip_processor, tokenizer, unk_token_id


In [13]:
def train_model():
    train_loader, val_loader, label_encoder, num_q_types, _, _, _ = prepare_data()
    model = ViVQATripleTaskModel(len(label_encoder.classes_), num_q_types).to(CONFIG['device'])

    qformer_params = [p for p in model.qformer.parameters() if p.requires_grad]
    query_token_params = [model.query_tokens] if model.query_tokens.requires_grad else []
    other_params = [p for n, p in model.named_parameters() if p.requires_grad and 'qformer' not in n and n != 'query_tokens']

    param_groups = [
        {'params': other_params, 'lr': CONFIG['lr']},
        {'params': qformer_params + query_token_params, 'lr': CONFIG['lr_qformer']},
    ]
    optimizer = torch.optim.AdamW(param_groups, weight_decay=CONFIG['weight_decay'])

    # --- SỬA CHỖ NÀY: Thay linear scheduler bằng custom CosineDecay ---
    total_steps = len(train_loader) * CONFIG['epochs']
    lr_end = 1e-6 # Target LR muốn giảm về ở cuối quá trình train
    
    cosine_scheduler_other = CosineDecay(CONFIG['lr'], lr_end, total_steps)
    cosine_scheduler_qformer = CosineDecay(CONFIG['lr_qformer'], lr_end, total_steps)
    global_step = 0
    # -----------------------------------------------------------------

    criterion_vqa = nn.CrossEntropyLoss(label_smoothing=0.1)
    criterion_qtype = nn.CrossEntropyLoss()
    criterion_bbox = nn.SmoothL1Loss(reduction='none')

    scaler = GradScaler()
    early_stopping = EarlyStopping(patience=CONFIG['patience'], path=CONFIG['save_path'])

    print(f"🚀 Bắt đầu training (80/20) với Cosine Decay...")
    
    for epoch in range(CONFIG['epochs']):
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

        for batch in pbar:
            # --- SỬA CHỖ NÀY: Cập nhật LR theo từng step ---
            global_step += 1
            optimizer.param_groups[0]['lr'] = cosine_scheduler_other.get_value(global_step)
            optimizer.param_groups[1]['lr'] = cosine_scheduler_qformer.get_value(global_step)
            # -----------------------------------------------

            pixel_values, input_ids, attention_mask = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16), batch['input_ids'].to(CONFIG['device']), batch['attention_mask'].to(CONFIG['device'])
            labels, q_types_gt = batch['labels'].to(CONFIG['device']), batch['q_type'].to(CONFIG['device'])
            target_box, has_box = batch['target_box'].to(CONFIG['device']), batch['has_box'].to(CONFIG['device'])

            optimizer.zero_grad()
            with autocast():
                ans_logits, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)
                loss_vqa, loss_qtype = criterion_vqa(ans_logits, labels), criterion_qtype(q_type_logits, q_types_gt)

                target_types = torch.tensor(CONFIG['box_required_types'], device=CONFIG['device'])
                color_mask = torch.isin(q_types_gt, target_types).float()
                valid_box_mask = (has_box == 1.0) & (color_mask == 1.0)

                loss_bbox_raw = criterion_bbox(pred_boxes, target_box).mean(dim=1)
                loss_bbox = (loss_bbox_raw * valid_box_mask).sum() / (valid_box_mask.sum() + 1e-6)
                loss_giou = giou_loss(pred_boxes[valid_box_mask].float(), target_box[valid_box_mask].float()) if valid_box_mask.any() else torch.tensor(0.0, device=CONFIG['device'])

                loss = loss_vqa + CONFIG['lambda_qtype'] * loss_qtype + CONFIG['lambda_bbox'] * loss_bbox + CONFIG['lambda_giou'] * loss_giou

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            # (Đã xóa dòng scheduler.step() cũ)
            
            # Thêm hiển thị LR để dễ quan sát
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'LR': f"{optimizer.param_groups[0]['lr']:.2e}"})

        # ---- VALIDATION ----
        model.eval()
        val_vqa_c, val_qtype_c, val_total, iou_sum, iou_count = 0, 0, 0, 0.0, 0

        with torch.no_grad(), autocast():
            for batch in val_loader:
                pixel_values, input_ids, attention_mask = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16), batch['input_ids'].to(CONFIG['device']), batch['attention_mask'].to(CONFIG['device'])
                labels, q_types_gt = batch['labels'].to(CONFIG['device']), batch['q_type'].to(CONFIG['device'])
                target_box, has_box = batch['target_box'].to(CONFIG['device']), batch['has_box'].to(CONFIG['device'])

                ans_logits, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)

                val_vqa_c += (ans_logits.argmax(1) == labels).sum().item()
                val_qtype_c += (q_type_logits.argmax(1) == q_types_gt).sum().item()
                val_total += labels.size(0)

                valid_box_mask = (has_box == 1.0)
                if valid_box_mask.any():
                    ious = calculate_iou(pred_boxes[valid_box_mask], target_box[valid_box_mask])
                    iou_sum += ious.sum().item(); iou_count += ious.size(0)

        val_acc_vqa, val_acc_qtype = val_vqa_c / val_total, val_qtype_c / val_total
        val_miou = (iou_sum / iou_count) if iou_count > 0 else 0.0
        combined_score = (val_acc_vqa + (val_miou * 1.5) + val_acc_qtype) / 3.5
        print(f"Epoch {epoch+1} | VQA: {val_acc_vqa:.4f} | Type: {val_acc_qtype:.4f} | mIoU: {val_miou:.4f} | Score: {combined_score:.4f}")

        early_stopping(combined_score, model, epoch)
        if early_stopping.early_stop:
            print("Early stopping: Mô hình đã hội tụ!")
            break

    return early_stopping.best_epoch

In [14]:
def train_full_data(best_epochs):
    print(f"\n🚀 BẮT ĐẦU RETRAIN TRÊN 100% DATA VỚI {best_epochs + 1} EPOCHS & COSINE DECAY...")
    
    blip_processor = Blip2Processor.from_pretrained(CONFIG['blip_model'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model'], use_fast=False)

    df_raw = pd.read_csv(CONFIG['train_csv'])
    if 'Unnamed: 0' in df_raw.columns:
        df_raw.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)

    df_box = pd.read_csv(CONFIG['train_box_csv'])
    if 'Unnamed: 0' in df_box.columns:
        df_box.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)
    
    df_merged = pd.merge(df_raw, df_box[['question_id', 'merged_box']], on='question_id', how='left')
    
    all_answers = df_merged['answer'].apply(lambda x: str(x).lower().strip()).unique().tolist()
    if 'unknown' not in all_answers: all_answers.append('unknown')
    label_encoder = LabelEncoder().fit(all_answers)
    unk_token_id = label_encoder.transform(['unknown'])[0]
    num_q_types = int(df_merged['type'].max() + 1)

    full_loader = DataLoader(
        ViVQATripleTaskDataset(df_merged, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id),
        batch_size=CONFIG['batch_size'], shuffle=True,
        num_workers=CONFIG['num_workers'], persistent_workers=True, pin_memory=True
    )

    model = ViVQATripleTaskModel(len(label_encoder.classes_), num_q_types).to(CONFIG['device'])
    
    other_params = [p for n, p in model.named_parameters() if p.requires_grad and 'qformer' not in n and n != 'query_tokens']
    qformer_params = [p for p in model.qformer.parameters() if p.requires_grad]
    param_groups = [{'params': other_params, 'lr': CONFIG['lr']}, {'params': qformer_params + [model.query_tokens], 'lr': CONFIG['lr_qformer']}]
    optimizer = torch.optim.AdamW(param_groups, weight_decay=CONFIG['weight_decay'])
    
    # --- SỬA CHỖ NÀY: Cosine Decay cho Full Train ---
    total_steps = len(full_loader) * (best_epochs + 1)
    lr_end = 1e-6
    
    cosine_scheduler_other = CosineDecay(CONFIG['lr'], lr_end, total_steps)
    cosine_scheduler_qformer = CosineDecay(CONFIG['lr_qformer'], lr_end, total_steps)
    global_step = 0
    # ------------------------------------------------

    criterion_vqa = nn.CrossEntropyLoss(label_smoothing=0.1)
    criterion_qtype = nn.CrossEntropyLoss()
    criterion_bbox = nn.SmoothL1Loss(reduction='none')
    scaler = GradScaler()

    for epoch in range(best_epochs + 1):
        model.train()
        pbar = tqdm(full_loader, desc=f"Full Train Epoch {epoch+1}/{best_epochs+1}")
        
        for batch in pbar:
            # --- SỬA CHỖ NÀY: Cập nhật LR ---
            global_step += 1
            optimizer.param_groups[0]['lr'] = cosine_scheduler_other.get_value(global_step)
            optimizer.param_groups[1]['lr'] = cosine_scheduler_qformer.get_value(global_step)
            # ---------------------------------
            
            pixel_values, input_ids, attention_mask = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16), batch['input_ids'].to(CONFIG['device']), batch['attention_mask'].to(CONFIG['device'])
            labels, q_types_gt = batch['labels'].to(CONFIG['device']), batch['q_type'].to(CONFIG['device'])
            target_box, has_box = batch['target_box'].to(CONFIG['device']), batch['has_box'].to(CONFIG['device'])

            optimizer.zero_grad()
            with autocast():
                ans_logits, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)
                loss_vqa, loss_qtype = criterion_vqa(ans_logits, labels), criterion_qtype(q_type_logits, q_types_gt)

                target_types = torch.tensor(CONFIG['box_required_types'], device=CONFIG['device'])
                color_mask = torch.isin(q_types_gt, target_types).float()
                valid_box_mask = (has_box == 1.0) & (color_mask == 1.0)

                loss_bbox_raw = criterion_bbox(pred_boxes, target_box).mean(dim=1)
                loss_bbox = (loss_bbox_raw * valid_box_mask).sum() / (valid_box_mask.sum() + 1e-6)
                loss_giou = giou_loss(pred_boxes[valid_box_mask].float(), target_box[valid_box_mask].float()) if valid_box_mask.any() else torch.tensor(0.0, device=CONFIG['device'])

                loss = loss_vqa + CONFIG['lambda_qtype']*loss_qtype + CONFIG['lambda_bbox']*loss_bbox + CONFIG['lambda_giou']*loss_giou

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            # (Đã xóa scheduler.step() cũ)
            
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'LR': f"{optimizer.param_groups[0]['lr']:.2e}"})

    full_save_path = CONFIG['save_path'].replace('.pth', '_100percent.pth')
    torch.save(model.state_dict(), full_save_path)
    print(f"✅ Đã lưu model train 100% data tại: {full_save_path}")

In [15]:
# ==========================================
# 6. INFERENCE CHIẾN LƯỢC THEO Q-TYPE
# ==========================================
def crop_and_reencode(image_pil, box_coords, blip_processor, min_size=32, pad_ratio=0.05):
    W, H = image_pil.size
    xmin, ymin, xmax, ymax = box_coords

    pad_x = (xmax - xmin) * pad_ratio
    pad_y = (ymax - ymin) * pad_ratio
    xmin, ymin = max(0.0, xmin - pad_x), max(0.0, ymin - pad_y)
    xmax, ymax = min(1.0, xmax + pad_x), min(1.0, ymax + pad_y)

    px0, py0 = int(xmin * W), int(ymin * H)
    px1, py1 = int(xmax * W), int(ymax * H)

    if (px1 - px0) < min_size or (py1 - py0) < min_size:
        cropped = image_pil
    else:
        cropped = image_pil.crop((px0, py0, px1, py1))

    return blip_processor(images=cropped, return_tensors="pt")['pixel_values']


def multi_strategy_inference_batch(model, batch, blip_processor, device, box_required_types=None, crop_weight=0.6):
    if box_required_types is None: box_required_types = CONFIG['box_required_types']

    pixel_values = batch['pixel_values'].to(device, dtype=torch.float16)
    input_ids, attention_mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)

    with torch.no_grad(), autocast():
        ans_logits_full, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)

    pred_qtypes = q_type_logits.argmax(1)
    ans_logits_final = ans_logits_full.clone()

    box_indices = [i for i, qt in enumerate(pred_qtypes.tolist()) if qt in box_required_types]

    if box_indices and 'raw_images' in batch:
        for i in box_indices:
            raw_img = batch['raw_images'][i]
            
            # CẬP NHẬT: Model bắt buộc phải tự dùng box do nó dự đoán (No GT Data Leak)
            box_coords = pred_boxes[i].cpu().tolist()
            effective_crop_weight = crop_weight

            crop_pixels = crop_and_reencode(raw_img, box_coords, blip_processor).to(device, dtype=torch.float16)

            with torch.no_grad(), autocast():
                ans_logits_crop, _, _ = model(crop_pixels, input_ids[i:i+1], attention_mask[i:i+1])

            ans_logits_final[i] = ((1 - effective_crop_weight) * ans_logits_full[i] + effective_crop_weight * ans_logits_crop.squeeze(0))

    pred_answers = ans_logits_final.argmax(1)
    return pred_answers, pred_boxes, pred_qtypes


In [16]:
# ==========================================
# 7. DATASET CHO INFERENCE (Có raw_images + GT box coords)
# ==========================================
class ViVQAInferenceDataset(ViVQATripleTaskDataset):
    """
    Dataset cho inference:
    - Thêm raw PIL image để crop
    - Thêm gt_box_coords (GT box normalized) nếu có từ test_box_csv
    """
    def __getitem__(self, idx):
        item = super().__getitem__(idx)
        row = self.data.iloc[idx]
        img_id = row.get('img_id', row.get('image'))
        img_path = find_image_path(self.image_dir, img_id)
        try:
            raw_image = Image.open(img_path).convert("RGB") if img_path else Image.new('RGB', (224, 224))
        except:
            raw_image = Image.new('RGB', (224, 224))

        item['raw_image'] = raw_image
        # gt_box_coords = target_box (đã được normalize ở parent class)
        item['gt_box_coords'] = item['target_box'].clone()
        return item

def inference_collate_fn(batch):
    raw_images = [item.pop('raw_image') for item in batch]
    collated = torch.utils.data.default_collate(batch)
    collated['raw_images'] = raw_images
    return collated


In [17]:
# ==========================================
# 8. TEST EVALUATION
# ==========================================
def test_evaluation():
    print(">>> Đang chuẩn bị dữ liệu TEST độc lập...")
    blip_processor = Blip2Processor.from_pretrained(CONFIG['blip_model'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model'], use_fast=False)

    df_raw = pd.read_csv(CONFIG['train_csv'])
    all_answers = df_raw['answer'].apply(lambda x: str(x).lower().strip()).unique().tolist()
    if 'unknown' not in all_answers: all_answers.append('unknown')
    label_encoder = LabelEncoder().fit(all_answers)
    unk_token_id = label_encoder.transform(['unknown'])[0]
    num_q_types = int(df_raw['type'].max() + 1)

    model = ViVQATripleTaskModel(len(label_encoder.classes_), num_q_types).to(CONFIG['device'])
    model.load_state_dict(torch.load(CONFIG['save_path'], map_location=CONFIG['device']))
    model.eval()

    test_df = pd.read_csv(CONFIG['test_csv'])
    if 'Unnamed: 0' in test_df.columns: test_df.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)

    if os.path.exists(CONFIG.get('test_box_csv', '')):
        df_box_test = pd.read_csv(CONFIG['test_box_csv'])
        if 'Unnamed: 0' in df_box_test.columns:
            df_box_test.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)
        test_df = pd.merge(test_df, df_box_test[['question_id', 'owl_box']], on='question_id', how='left')
        n_with_box = test_df['owl_box'].notna().sum()
        print(f"  Số mẫu test có GT box: {n_with_box}/{len(test_df)}")

    test_dataset = ViVQAInferenceDataset(
        test_df, CONFIG['test_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id
    )
    test_loader = DataLoader(
        test_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
        num_workers=CONFIG['num_workers'], persistent_workers=True,
        collate_fn=inference_collate_fn
    )

    print(">>> Đang chạy dự đoán trên tập TEST...")
    test_vqa_c, test_qtype_c, test_total = 0, 0, 0
    iou_sum, iou_count = 0.0, 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            labels = batch['labels'].to(CONFIG['device'])
            q_types_gt = batch['q_type'].to(CONFIG['device'])
            target_box = batch['target_box'].to(CONFIG['device'])
            has_box = batch['has_box'].to(CONFIG['device'])

            pred_answers, pred_boxes, pred_qtypes = multi_strategy_inference_batch(
                model, batch, blip_processor, CONFIG['device'],
                box_required_types=CONFIG['box_required_types'],
                crop_weight=0.6,
                pad_ratio=0.05
            )

            test_vqa_c += (pred_answers == labels).sum().item()
            test_qtype_c += (pred_qtypes == q_types_gt).sum().item()
            test_total += labels.size(0)

            valid_box_mask = (has_box == 1.0)
            if valid_box_mask.any():
                ious = calculate_iou(pred_boxes[valid_box_mask], target_box[valid_box_mask])
                iou_sum += ious.sum().item()
                iou_count += ious.size(0)

    test_acc_vqa = test_vqa_c / test_total
    test_acc_qtype = test_qtype_c / test_total
    test_miou = (iou_sum / iou_count) if iou_count > 0 else 0.0

    print("\n" + "="*40)
    print("KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST ĐỘC LẬP:")
    print(f"- Độ chính xác VQA:        {test_acc_vqa*100:.2f}%")
    print(f"- Độ chính xác đoán Type:  {test_acc_qtype*100:.2f}%")
    if iou_count > 0:
        print(f"- Mean IoU Bounding Box:   {test_miou*100:.2f}%")
    else:
        print("- Bounding Box: Không có dữ liệu Ground-truth để đối chiếu.")
    print("="*40)


In [18]:
# ==========================================
# 9. ĐÁNH GIÁ CHI TIẾT THEO Q-TYPE
# ==========================================
def evaluate_and_print_details():
    print(">>> Đang chuẩn bị dữ liệu TEST...")
    blip_processor = Blip2Processor.from_pretrained(CONFIG['blip_model'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model'], use_fast=False)

    df_raw = pd.read_csv(CONFIG['train_csv'])
    all_answers = df_raw['answer'].apply(lambda x: str(x).lower().strip()).unique().tolist()
    if 'unknown' not in all_answers: all_answers.append('unknown')
    label_encoder = LabelEncoder().fit(all_answers)
    unk_token_id = label_encoder.transform(['unknown'])[0]
    num_q_types = int(df_raw['type'].max() + 1)

    model = ViVQATripleTaskModel(len(label_encoder.classes_), num_q_types).to(CONFIG['device'])
    model.load_state_dict(torch.load(CONFIG['save_path'], map_location=CONFIG['device']))
    model.eval()

    test_df = pd.read_csv(CONFIG['test_csv'])
    if 'Unnamed: 0' in test_df.columns: test_df.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)

    if os.path.exists(CONFIG.get('test_box_csv', '')):
        df_box_test = pd.read_csv(CONFIG['test_box_csv'])
        if 'Unnamed: 0' in df_box_test.columns:
            df_box_test.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)
        test_df = pd.merge(test_df, df_box_test[['question_id', 'owl_box']], on='question_id', how='left')

    test_dataset = ViVQAInferenceDataset(
        test_df, CONFIG['test_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id
    )
    test_loader = DataLoader(
        test_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
        num_workers=CONFIG['num_workers'], persistent_workers=True,
        collate_fn=inference_collate_fn
    )

    print(">>> Đang chạy dự đoán...")
    test_vqa_c, test_qtype_c, test_total = 0, 0, 0
    iou_sum, iou_count = 0.0, 0
    correct_per_type = {}
    total_per_type = {}
    box_type_correct = {}
    box_type_total = {}

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing & Evaluating"):
            labels = batch['labels'].to(CONFIG['device'])
            q_types_gt = batch['q_type'].to(CONFIG['device'])
            target_box = batch['target_box'].to(CONFIG['device'])
            has_box = batch['has_box'].to(CONFIG['device'])

            pred_answers, pred_boxes, pred_qtypes = multi_strategy_inference_batch(
                model, batch, blip_processor, CONFIG['device'],
                box_required_types=CONFIG['box_required_types'],
                crop_weight=0.6,
                pad_ratio=0.05
            )

            test_vqa_c += (pred_answers == labels).sum().item()
            test_qtype_c += (pred_qtypes == q_types_gt).sum().item()
            test_total += labels.size(0)

            valid_box_mask = (has_box == 1.0)
            if valid_box_mask.any():
                ious = calculate_iou(pred_boxes[valid_box_mask], target_box[valid_box_mask])
                iou_sum += ious.sum().item()
                iou_count += ious.size(0)

            for pred, target, q_type in zip(pred_answers, labels, q_types_gt):
                q_t = q_type.item()
                if q_t not in total_per_type:
                    total_per_type[q_t] = 0
                    correct_per_type[q_t] = 0
                total_per_type[q_t] += 1
                if pred.item() == target.item():
                    correct_per_type[q_t] += 1

                strategy = 'box_guided' if q_t in CONFIG['box_required_types'] else 'standard'
                if strategy not in box_type_total:
                    box_type_total[strategy] = 0
                    box_type_correct[strategy] = 0
                box_type_total[strategy] += 1
                if pred.item() == target.item():
                    box_type_correct[strategy] += 1

    test_acc_vqa = test_vqa_c / test_total
    test_acc_qtype = test_qtype_c / test_total
    test_miou = (iou_sum / iou_count) if iou_count > 0 else 0.0

    print("\n" + "="*55)
    print(" KẾT QUẢ TỔNG QUAN TRÊN TẬP TEST ĐỘC LẬP")
    print("="*55)
    print(f"- Độ chính xác VQA:        {test_acc_vqa*100:.2f}%")
    print(f"- Độ chính xác đoán Type:  {test_acc_qtype*100:.2f}%")
    if iou_count > 0:
        print(f"- Mean IoU Bounding Box:   {test_miou*100:.2f}%")
    else:
        print("- Bounding Box: Không có dữ liệu Ground-truth để đối chiếu.")

    print("\n" + "="*55)
    print(" ĐỘ CHÍNH XÁC THEO CHIẾN LƯỢC INFERENCE")
    print("="*55)
    for strategy in ['standard', 'box_guided']:
        if strategy in box_type_total and box_type_total[strategy] > 0:
            acc = box_type_correct[strategy] / box_type_total[strategy] * 100
            types_in = CONFIG['box_required_types'] if strategy == 'box_guided' else 'others'
            print(f"- {strategy:12s} (type={types_in}): {acc:.2f}%  | {box_type_total[strategy]} câu")

    print("\n" + "="*55)
    print(" ĐỘ CHÍNH XÁC VQA THEO TỪNG LOẠI CÂU HỎI (Q-TYPE)")
    print("="*55)
    for q_t in sorted(total_per_type.keys()):
        acc = (correct_per_type[q_t] / total_per_type[q_t]) * 100
        strategy_label = " [box_guided]" if q_t in CONFIG['box_required_types'] else " [standard]"
        print(f"- Type {q_t}{strategy_label}: {acc:.2f}%  | Đúng: {correct_per_type[q_t]}/{total_per_type[q_t]} câu")
    print("="*55)


In [19]:
import gc

# Chạy xong train_model
best_epoch_found = train_model()

# DỌN DẸP BỘ NHỚ TRƯỚC KHI CHẠY HÀM TIẾP THEO
torch.cuda.empty_cache()
gc.collect()

# Chạy full train
train_full_data(best_epoch_found)


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

...

Epoch 12 | VQA: 0.4708 | Type: 0.4154 | mIoU: 0.5307 | Score: 0.4806


Epoch 13:   0%|          | 0/300 [00:00<?, ?it/s]

Epoch 13 | VQA: 0.4537 | Type: 0.4154 | mIoU: 0.5124 | Score: 0.4679


Epoch 14:   0%|          | 0/300 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:

# DỌN DẸP TIẾP
torch.cuda.empty_cache()
gc.collect()

test_evaluation()
torch.cuda.empty_cache()
gc.collect()

evaluate_and_print_details()